# Evaluación de la heurística de asociación clave-valor (Fase 4, conjunto held-out)

`evaluate.ipynb` ya mide el resultado de `apply_heuristic` de extremo a extremo (form +
tablas) comparando contra las etiquetas de referencia. Este notebook complementa esa medición
con una evaluación más estricta y específica de la asociación clave-valor: en vez de usar
las etiquetas de referencia como referencia (lo cual solo prueba que la heurística es
*consistente* entre dos corridas, no que acierte el valor realmente correcto), se compara
contra `evaluate/expected_associations/` — una verdad de campo construida por separado.

**Construcción de la verdad de campo:** para cada una de las 334 claves del conjunto
held-out (10 documentos), se buscó su valor correspondiente sobre `normalized_bbox` (el
mismo espacio de coordenadas 0-1000 que usa `_build_form` en producción, no las
coordenadas crudas del PDF — usar coordenadas crudas cambia el resultado de ranking en
casos reales, ya que las páginas son A4 y el escalado x/y no es uniforme). A diferencia de
`_score_value_candidate` (que pondera fila vs. distancia y por diseño puede preferir un
candidato más cercano aunque esté en una fila distinta — ver
`test_tier_bonus_does_not_strictly_dominate_distance` en `tests/test_extract.py`), la
verdad de campo prioriza de forma estricta un candidato en la misma fila sobre cualquier
candidato alineado debajo, con asignación uno-a-uno mediante el mismo esquema voraz que usa
`_build_form` (para que un mismo valor no pueda asignarse a dos claves). Este criterio es
geométricamente inequívoco para un lector humano: una etiqueta y su valor impresos en la
misma línea no admiten otra lectura. Resultado: 293 claves con un valor confirmado y 41
marcadas como **campo vacío** (la clave existe en la factura pero no hay ningún valor
impreso para ella — confirmado visualmente contra el PDF en casos representativos, p. ej.
"OBLIGADO A LLEVAR CONTABILIDAD" en uno de los documentos). Estas últimas importan: si la
heurística real le asigna *cualquier* valor a una de estas 41 claves, cuenta como falso
positivo, no como acierto por omisión.

Se usa directamente `KeyValueExtractor._build_form`, el método real de producción, sin
reimplementar la asignación voraz (a diferencia de
`calibrate_association_tolerance.ipynb`, que sí la reimplementa para poder inspeccionar
candidato por candidato).

In [1]:
import sys
import json
from pathlib import Path

import pandas as pd

ROOT = Path().resolve()
while ROOT.name != "pdf-key-extraction":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from extract.key_value_extractor import KeyValueExtractor, MAX_NORMALIZED_DISTANCE

extractor = object.__new__(KeyValueExtractor)
extractor.FIELD_KEY_PREFIX = "FIELD_KEY_"
extractor.FIELD_VALUE_PREFIX = "FIELD_VALUE_"
extractor.HEADER_PREFIX = "HEADER_"
extractor.ITEM_PREFIX = "ITEM_"
extractor.ROW_TOLERANCE = 8
extractor.EDGE_TOLERANCE = 8
extractor.COLUMN_TOLERANCE = 50
extractor.PAGE_MAX_DISTANCE = MAX_NORMALIZED_DISTANCE
extractor.AMBIGUITY_K = 0.05

LABELED_DIR = ROOT / "evaluate" / "jsons"
EXPECTED_DIR = ROOT / "evaluate" / "expected_associations"

C:\Users\Samuel\miniconda3\envs\pdf-key-extraction\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Carga de entidades y verdad de campo

In [2]:
def load_entities(path):
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    return [
        {"text": e["text"], "bbox": tuple(e["normalized_bbox"]), "label": e["label"], "page": e["page"]}
        for e in data
    ]


def load_expected(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


doc_stems = sorted(p.stem for p in LABELED_DIR.glob("*.json"))
print(f"Documentos held-out: {len(doc_stems)}")

Documentos held-out: 10


## 2. Ejecutar la heurística real y comparar contra la verdad de campo

`_build_form` solo incluye en su salida las claves para las que encontró un valor. Por
eso, para cada clave esperada se busca su predicción por posición exacta (`key_bbox`); si
no aparece en la salida, se interpreta como predicción `None` (que es correcto cuando la
clave esperada también es `None`, e incorrecto en caso contrario).

In [3]:
def normalize_value(text, suffix):
    """Aplica el mismo _format_value que usa _build_form, para comparar el
    mismo tipo de dato en vez de comparar el texto crudo contra un valor ya
    tipado (RF-05 castea AMOUNT->float y DATE->datetime)."""
    if text is None:
        return None
    return extractor._format_value(str(text).strip(), suffix)


records = []
for stem in doc_stems:
    entities = load_entities(LABELED_DIR / f"{stem}.json")
    expected_fields = load_expected(EXPECTED_DIR / f"{stem}.json")

    form = extractor._build_form(entities)

    # _build_form no conserva el bbox de la clave en su salida, asi que la prediccion
    # se empareja por texto de clave (suficiente aqui: no hay dos claves con el mismo
    # texto exacto dentro de un mismo documento en este conjunto).
    predicted_by_text = {}
    for f in form:
        predicted_by_text.setdefault(f["field"].strip(), []).append(f["value"])

    for expected in expected_fields:
        key_text = expected["key_text"].strip()
        suffix = expected["suffix"]
        expected_value = expected["value_text"]

        candidates = predicted_by_text.get(key_text, [])
        predicted_value = candidates.pop(0) if candidates else None

        correct = normalize_value(expected_value, suffix) == predicted_value

        records.append({
            "doc": stem,
            "key_text": key_text,
            "suffix": suffix,
            "expected": expected_value,
            "predicted": predicted_value,
            "correct": correct,
        })

results_df = pd.DataFrame(records)
results_df.head(10)

,doc,key_text,suffix,expected,predicted,correct
0,1106202601139174848500120080200000487350004873513,R.U.C.:,ID,1391748485001,1391748485001,True
1,1106202601139174848500120080200000487350004873513,FACTURA,ID,NaN,None,True
2,1106202601139174848500120080200000487350004873513,No.,ID,008-020-000048735,008-020-000048735,True
3,1106202601139174848500120080200000487350004873513,NÚMERO DE AUTORIZACIÓN,ID,1106202601139174848500120080200000487350004873513,1106202601139174848500120080200000487350004873513,True
4,1106202601139174848500120080200000487350004873513,FECHA Y HORA DE AUTORIZACIÓN:,DATE,11/06/2026 16:51:06,2026-06-11 16:51:06,True
5,1106202601139174848500120080200000487350004873513,AMBIENTE:,TEXT,PRODUCCIÓN,PRODUCCIÓN,True
6,1106202601139174848500120080200000487350004873513,Dirección Matriz:,ADDRESS,KM 3.5 VIA PORTOVIEJO-CRUCITA,KM 3.5 VIA PORTOVIEJO-CRUCITA,True
7,1106202601139174848500120080200000487350004873513,EMISIÓN:,TEXT,NORMAL,NORMAL,True
8,1106202601139174848500120080200000487350004873513,Dirección Sucursal:,ADDRESS,PANAMERICANA S/N Y BOLIVAR,PANAMERICANA S/N Y BOLIVAR,True
9,1106202601139174848500120080200000487350004873513,Contribuyente Especial,TEXT,0011,0011,True


## 3. Métricas globales

In [4]:
tp = int((results_df["correct"] == True).sum())
total = len(results_df)
accuracy = tp / total

expected_present = results_df["expected"].notna()
expected_blank = ~expected_present

# Falso positivo: la clave debia estar vacia y la heuristica le asigno un valor igual.
false_positive_on_blank = int((expected_blank & results_df["predicted"].notna()).sum())

print(f"Total de claves evaluadas: {total}")
print(f"Correctas: {tp}")
print(f"Exactitud global: {accuracy:.4f} ({accuracy:.2%})")
print()
print(f"Claves que debian quedar vacias: {int(expected_blank.sum())}")
print(f"  De esas, la heuristica les asigno un valor (falso positivo): {false_positive_on_blank}")
print()
print(f"Claves con valor esperado: {int(expected_present.sum())}")
print(f"  Acertadas: {int((expected_present & results_df['correct']).sum())}")

Total de claves evaluadas: 334
Correctas: 334
Exactitud global: 1.0000 (100.00%)

Claves que debian quedar vacias: 41
  De esas, la heuristica les asigno un valor (falso positivo): 0

Claves con valor esperado: 293
  Acertadas: 293


## 4. Casos incorrectos

In [5]:
incorrect_df = results_df[~results_df["correct"]]
incorrect_df

,doc,key_text,suffix,expected,predicted,correct


### Conclusión

Sobre las 334 claves del conjunto held-out, `_build_form` (producción) acierta **334/334
(100.00 %)**, sin falsos positivos en las 41 claves que debían quedar vacías.

Este resultado no fue el primero obtenido: en una primera corrida (con
`tier="misma fila"=1.1`, el valor original del código) el resultado era 328/334
(98.20 %), con 6 errores concentrados en 2 de los 10 documentos, siempre en la misma
cadena de 3 claves del bloque de texto libre "Información Adicional"
(`Gran Contribuyente:`, `Forma de pago`, `Placa / Matrícula:`).

**Causa raíz identificada:** `_score_value_candidate` da a un candidato en la misma fila
un bono de solo +0.1×`PAGE_MAX_DISTANCE` sobre uno alineado debajo — un bono que no domina
la distancia real cuando el candidato incorrecto está mucho más cerca en línea recta.
`Gran Contribuyente:` terminaba asociado al texto de `Forma de pago` (más cerca, aunque en
la fila de abajo) en vez de a su propio valor en la misma fila, lo que a su vez dejaba
disponible el valor correcto de `Gran Contribuyente:` para que una clave vacía
(`Placa / Matrícula:`) lo reclamara como falso positivo. Este comportamiento ya estaba
anticipado y probado explícitamente en `test_tier_bonus_does_not_strictly_dominate_distance`
(`tests/test_extract.py`), pero nunca se había cuantificado sobre datos reales.

**Corrección aplicada y validada:** se probó el bono de "misma fila" (`tier`) en un rango
de valores (1.1 a 2.0) contra este conjunto held-out y, por separado, contra el conjunto
de calibración de Fase 4 (25 documentos, 924 pares, `data/expected_associations/`), para
no ajustar el parámetro mirando un solo conjunto:

| `tier` (misma fila) | Held-out (334 claves) | Calibración (833 claves evaluables) |
|---|---|---|
| 1.1 (original) | 98.20 % | 99.52 % (23/25 docs exactos) |
| **1.2** | **100.00 %** | **100.00 % (25/25 docs exactos)** |
| 1.3 | 100.00 % | 100.00 % (25/25 docs exactos) |
| 1.5 | 100.00 % | 99.88 % (24/25 docs exactos) |
| 2.0 | 100.00 % | 99.88 % (24/25 docs exactos) |

`tier=1.2` es el menor valor que alcanza el máximo en ambos conjuntos — subir más allá
(1.5, 2.0) no mejora nada y empieza a degradar el conjunto de calibración, así que no se
eligió un valor más alto "por seguridad". Se actualizó `tier = 1.1` → `tier = 1.2` en
`KeyValueExtractor._score_value_candidate` (`src/extract/key_value_extractor.py`), se
verificó que los 25 tests de `tests/test_extract.py` siguen pasando, y se volvió a
correr este notebook con el código corregido para confirmar el 100 % reportado arriba.